# Colab T4 Temporal Neural Network Experiments

Use this notebook in Google Colab with a T4 GPU. Upload the repository and official Kaggle files, then run the cells. This is designed for serious neural experiments that are too slow on local CPU.

## Colab Setup

1. Runtime > Change runtime type > T4 GPU.
2. Upload or clone the repository.
3. Put `train_raw.csv`, `test_raw.csv`, `test.csv`, and `sample_submission.csv` under `data/raw/`.
4. Run the cells below.

In [ ]:
!nvidia-smi
!pip -q install lightgbm xgboost catboost

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    # Change this if your repo is mounted elsewhere in Colab.
    ROOT = Path('/content/CO5420-Air-Pollution-Forecasting_G26')
sys.path.append(str(ROOT))
ROOT

## Strong Baseline First

Always regenerate the best LightGBM baseline before training neural models. Neural models must beat this, not just persistence.

In [ ]:
from src.modern_temporal_rmse_improvements import run as run_modern

run_modern(data_dir=ROOT / 'data' / 'raw', output_dir=ROOT, train_fraction=0.8, random_state=42)

## GPU Neural Plan

Run the existing temporal neural module with more epochs and larger training rows. On local CPU this was too slow; on T4 this is the correct place to test it.

In [ ]:
!python -m src.temporal_neural_models \
  --data-dir data/raw \
  --output-dir . \
  --epochs 25 \
  --batch-size 512 \
  --max-train-rows 220000

## Decision Rule

Do not submit a neural model unless it beats the compact LightGBM on chronological validation and diagnostic official-test analysis. If it is close but different, use it only as a small blend component chosen from chronological validation, not from public leaderboard guessing.